# py_key_value_aio smoke test

Async key-value store with pluggable backends. The package's import name is `key_value.aio`.
Many backend submodules (redis, mongodb, dynamodb, ...) require optional `extras` that may not
be installed; the smoke-test cell tolerates those import failures.

In [ ]:
# Submodule smoke test: import the package + walk every submodule.
import importlib
import pkgutil

import key_value.aio

print(f"key_value.aio.__path__ = {key_value.aio.__path__}")

failures = []
for module_info in pkgutil.walk_packages(key_value.aio.__path__, prefix='key_value.aio.'):
    try:
        importlib.import_module(module_info.name)
    except Exception as e:
        failures.append((module_info.name, type(e).__name__, str(e)[:200]))

# Backend submodules (redis, mongodb, aerospike, ...) raise ImportError when extras are absent.
hard = [f for f in failures if f[1] not in ('ImportError', 'ModuleNotFoundError')]
if hard:
    print(f"\nHARD failures ({len(hard)}):")
    for name, exc_type, msg in hard[:20]:
        print(f"  {name}: {exc_type}: {msg}")
else:
    print(f"\nAll submodules import cleanly or fail only on missing optional extras "
          f"({len(failures)} optional-extra ImportError(s) tolerated)")
assert not hard, f"unexpected import failures: {hard}"

In [ ]:
# Example 1: SimpleStore — basic put/get/delete round-trip on the in-tree dict-backed store.
# SimpleStore is the dependency-free reference backend (no cachetools/redis/etc.).
from key_value.aio.stores.simple import SimpleStore

store = SimpleStore(default_collection='default')
await store.put('user:1', {'name': 'Alice', 'age': 30})

value = await store.get('user:1')
print(f"get user:1 -> {value}")
assert value == {'name': 'Alice', 'age': 30}

missing = await store.get('user:does-not-exist')
print(f"get missing -> {missing}")
assert missing is None

deleted = await store.delete('user:1')
print(f"delete user:1 -> {deleted}")
assert deleted is True
assert await store.get('user:1') is None

In [ ]:
# Example 2: bulk put_many / get_many / delete_many on SimpleStore.
from key_value.aio.stores.simple import SimpleStore

store = SimpleStore(default_collection='users')
keys = ['k1', 'k2', 'k3']
values = [{'i': 1}, {'i': 2}, {'i': 3}]
await store.put_many(keys=keys, values=values)

got = await store.get_many(keys=keys + ['missing'])
print(f"get_many -> {got}")
assert got[:3] == values
assert got[3] is None

n_deleted = await store.delete_many(keys=['k1', 'k2', 'missing'])
print(f"delete_many -> {n_deleted}")
assert n_deleted == 2

remaining = await store.get_many(keys=keys)
print(f"remaining -> {remaining}")
assert remaining == [None, None, {'i': 3}]

In [ ]:
# Example 3: TTL — put with ttl=10, observe a non-None ttl_remaining via .ttl().
from key_value.aio.stores.simple import SimpleStore

store = SimpleStore(default_collection='sessions')
await store.put('session:abc', {'user_id': 42}, ttl=10.0)

value, ttl_remaining = await store.ttl('session:abc')
print(f"value -> {value}")
print(f"ttl_remaining -> {ttl_remaining}")
assert value == {'user_id': 42}
assert ttl_remaining is not None and 0.0 < ttl_remaining <= 10.0

# An entry stored without ttl reports None for the ttl component.
await store.put('perm:xyz', {'permanent': True})
_, perm_ttl = await store.ttl('perm:xyz')
print(f"perm_ttl -> {perm_ttl}")
assert perm_ttl is None

In [ ]:
# Example 4: collections — multiple named collections coexist; enumerate keys + names.
from key_value.aio.stores.simple import SimpleStore

store = SimpleStore()
await store.put('a', {'v': 1}, collection='alpha')
await store.put('b', {'v': 2}, collection='alpha')
await store.put('c', {'v': 3}, collection='beta')

alpha_keys = sorted(await store.keys(collection='alpha'))
beta_keys = sorted(await store.keys(collection='beta'))
print(f"alpha keys -> {alpha_keys}")
print(f"beta keys  -> {beta_keys}")
assert alpha_keys == ['a', 'b']
assert beta_keys == ['c']

collections = sorted(await store.collections())
print(f"collections -> {collections}")
assert {'alpha', 'beta'}.issubset(set(collections))

In [ ]:
# Example 5: NullStore + PrefixKeysWrapper — wrapper composition + protocol check.
from key_value.aio.stores.null import NullStore
from key_value.aio.stores.simple import SimpleStore
from key_value.aio.wrappers.prefix_keys import PrefixKeysWrapper
from key_value.aio.protocols.key_value import AsyncKeyValueProtocol

# NullStore drops everything (the /dev/null backend).
null = NullStore()
await null.put('k', {'v': 1})
assert await null.get('k') is None

# PrefixKeysWrapper delegates to its underlying store under a prefixed key.
simple = SimpleStore(default_collection='wrapped')
prefixed = PrefixKeysWrapper(key_value=simple, prefix='app::')
await prefixed.put('item', {'payload': 'hello'})

# The wrapper round-trips its own logical key...
via_wrapper = await prefixed.get('item')
print(f"via_wrapper={via_wrapper}")
assert via_wrapper == {'payload': 'hello'}

# ...while the underlying store never sees the bare key — it is prefixed.
assert await simple.get('item') is None
underlying_keys = await simple.keys(collection='wrapped')
print(f"underlying stored keys={underlying_keys}")
assert any(k.startswith('app::') for k in underlying_keys)

assert isinstance(simple, AsyncKeyValueProtocol)
assert isinstance(prefixed, AsyncKeyValueProtocol)
print("py_key_value_aio smoke test PASS")